In [ ]:
# ===============================================
# BASELINE CONTAMINATION MODELING (NO CUSTOM ALGORITHM)
# Target: Single compound value >= 10.0
# ===============================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import joblib

print("Loading PFAS Data...")
BASE_DIR = Path.cwd()
if (BASE_DIR / 'pfas_major_compounds.csv').exists():
    csv_path = BASE_DIR / 'pfas_major_compounds.csv'
else:
    csv_path = BASE_DIR / 'notebooks' / 'pfas_major_compounds.csv'

df = pd.read_csv(csv_path)

# Define baseline target (Value >= 10.0)
target_col = 'value' 
THRESHOLD = 10.0 
df['is_contaminated'] = (df[target_col] >= THRESHOLD).astype(int)

print(f"Class Distribution:\n{df['is_contaminated'].value_counts(normalize=True) * 100}")

# Base features
lat_col = 'lat'
lon_col = 'lon'
df = df.dropna(subset=[lat_col, lon_col])

# Dummy encode media
df = pd.get_dummies(df, columns=['type'], drop_first=True)
features = [lat_col, lon_col] + [c for c in df.columns if 'type_' in c]

model_df = df.dropna(subset=features + ['is_contaminated']).copy()
X = model_df[features]
y = model_df['is_contaminated']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- TRAINING 4 BASELINE MODELS ---")

# 1. Extra Trees
print("Training Extra Trees...")
base_et = ExtraTreesClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
base_et.fit(X_train, y_train)

# 2. KNN
print("Training KNN...")
base_knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
base_knn.fit(X_train_scaled, y_train)

# 3. Random Forest
print("Training Random Forest...")
base_rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
base_rf.fit(X_train, y_train)

# 4. XGBoost
print("Training XGBoost...")
pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
base_xgb = XGBClassifier(scale_pos_weight=pos_weight, n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
base_xgb.fit(X_train, y_train)

print("\n--- BASELINE RESULTS ---")
results = []
results.append(('Extra Trees', roc_auc_score(y_test, base_et.predict_proba(X_test)[:, 1]), accuracy_score(y_test, base_et.predict(X_test))))
results.append(('KNN', roc_auc_score(y_test, base_knn.predict_proba(X_test_scaled)[:, 1]), accuracy_score(y_test, base_knn.predict(X_test_scaled))))
results.append(('Random Forest', roc_auc_score(y_test, base_rf.predict_proba(X_test)[:, 1]), accuracy_score(y_test, base_rf.predict(X_test))))
results.append(('XGBoost', roc_auc_score(y_test, base_xgb.predict_proba(X_test)[:, 1]), accuracy_score(y_test, base_xgb.predict(X_test))))

results.sort(key=lambda x: x[1], reverse=True)
for rank, (name, auc, acc) in enumerate(results):
    print(f"#{rank+1} -> {name:20s} | ROC-AUC: {auc:.4f}  | Accuracy: {acc:.4f}")


 Contamination Modeling Engine





In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import BallTree


print("Loading PFAS Data...")

df = pd.read_csv('pfas_major_compounds.csv')

print(f"Data Loaded! Shape: {df.shape}")


try:
    hotspots_df = pd.read_parquet('../data/processed/pfas_hotspots.parquet')
    print("Hotspots loaded.")
except FileNotFoundError:
    print("Hotspots file not found. Distance features will be skipped or need generation.")
    hotspots_df = None



Loading PFAS Data...
Data Loaded! Shape: (369264, 16)
Hotspots file not found. Distance features will be skipped or need generation.


 Define Target Variables (Labels)


In [2]:
THRESHOLD = 10.0 
val_cols = [c for c in df.columns if 'conc' in c.lower() or 'val' in c.lower() or 'result' in c.lower()]

if len(val_cols) > 0:
    target_col = val_cols[0]
    print(f"Using '{target_col}' as the concentration measurement column.")
else:
    
    target_col = 'Concentration' 
    print(f"Replace '{target_col}' with your actual measurement column name. Columns available: {df.columns.tolist()}")


df['is_contaminated'] = (df[target_col] >= THRESHOLD).astype(int)

print(f"Class Distribution: \n{df['is_contaminated'].value_counts(normalize=True) * 100}%")



Using 'value' as the concentration measurement column.
Class Distribution: 
is_contaminated
0    60.465412
1    39.534588
Name: proportion, dtype: float64%


 Feature Engineering (Geospatial Inputs)


In [3]:

lat_col = [c for c in df.columns if 'lat' in c.lower()][0] if len([c for c in df.columns if 'lat' in c.lower()]) > 0 else 'Latitude'
lon_col = [c for c in df.columns if 'lon' in c.lower() or 'lng' in c.lower()][0] if len([c for c in df.columns if 'lon' in c.lower() or 'lng' in c.lower()]) > 0 else 'Longitude'


df = df.dropna(subset=[lat_col, lon_col])


if hotspots_df is not None:
    hotspot_lat = [c for c in hotspots_df.columns if 'lat' in c.lower()][0]
    hotspot_lon = [c for c in hotspots_df.columns if 'lon' in c.lower()][0]
    
    
    hotspots_df = hotspots_df.dropna(subset=[hotspot_lat, hotspot_lon])
    
   
    tree = BallTree(np.deg2rad(hotspots_df[[hotspot_lat, hotspot_lon]].values), metric='haversine')
    
    query_points = np.deg2rad(df[[lat_col, lon_col]].values)
    dist, _ = tree.query(query_points, k=1)
    
  
    df['dist_to_nearest_hotspot_km'] = dist.flatten() * 6371.0
else:
    df['dist_to_nearest_hotspot_km'] = 0.0

features = [lat_col, lon_col, 'dist_to_nearest_hotspot_km']

media_cols = [c for c in df.columns if 'media' in c.lower() or 'matrix' in c.lower()]
if media_cols:
    media_col = media_cols[0]
    df = pd.get_dummies(df, columns=[media_col], drop_first=True)
    features.extend([c for c in df.columns if media_col in c])

model_df = df.dropna(subset=features + ['is_contaminated']).copy()
X = model_df[features]
y = model_df['is_contaminated']


 Geographic Train-Test Validated Split


In [4]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training Data: {X_train.shape[0]} records")
print(f"Testing Data: {X_test.shape[0]} records")



Training Data: 276948 records
Testing Data: 92316 records


 Model Training Pipeline and Evaluation



In [5]:

print("--- Training Logistic Regression (Baseline) ---")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train, y_train)


lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]


print("Classification Report:")
print(classification_report(y_test, lr_preds))
print(f"ROC-AUC Score: {roc_auc_score(y_test, lr_probs):.4f}\n")


print("--- Training XGBoost Classifier ---")
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss', scale_pos_weight=(y_test.value_counts()[0]/max(y_test.value_counts()[1],1)))
xgb_model.fit(X_train, y_train)


xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]


print("Classification Report:")
print(classification_report(y_test, xgb_preds))
print(f"ROC-AUC Score: {roc_auc_score(y_test, xgb_probs):.4f}\n")



--- Training Logistic Regression (Baseline) ---
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.68      0.71     55819
           1       0.57      0.64      0.60     36497

    accuracy                           0.67     92316
   macro avg       0.66      0.66      0.66     92316
weighted avg       0.68      0.67      0.67     92316

ROC-AUC Score: 0.6885

--- Training XGBoost Classifier ---
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.85      0.86     55819
           1       0.77      0.79      0.78     36497

    accuracy                           0.83     92316
   macro avg       0.82      0.82      0.82     92316
weighted avg       0.83      0.83      0.83     92316

ROC-AUC Score: 0.8971



### Results Visualization


In [6]:

importance = xgb_model.feature_importances_
feature_names = X.columns
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df)
plt.title('XGBoost Feature Importance for Contamination Likelihood')
plt.show()



In [7]:
import numpy as np

print("Generating Composite Contamination Index (Module 5)...")


target_col = [c for c in df.columns if 'conc' in c.lower() or 'val' in c.lower() or 'result' in c.lower()][0]


safe_conc = df[target_col].clip(lower=0)
log_conc = np.log10(safe_conc + 1)


log_range = log_conc.max() - log_conc.min()
if log_range == 0:
    df['Concentration_Score'] = 0.0
else:
    df['Concentration_Score'] = (log_conc - log_conc.min()) / log_range


max_dist = df['dist_to_nearest_hotspot_km'].max()


if max_dist == 0:
   
    df['Spatial_Density_Score'] = 1.0
else:
    df['Spatial_Density_Score'] = 1.0 - (df['dist_to_nearest_hotspot_km'] / max_dist)

df['Chemical_Weight'] = 1.0 


df['Composite_Contamination_Index'] = (
    (df['Concentration_Score'] * 0.60) + 
    (df['Spatial_Density_Score'] * 0.40)
) * 100 * df['Chemical_Weight']


df['Composite_Contamination_Index'] = df['Composite_Contamination_Index'].fillna(0).clip(upper=100.0)


def classify_tier(score):
    if score >= 60: return "High"
    elif score >= 25: return "Moderate"
    else: return "Low"

df['Contamination_Tier'] = df['Composite_Contamination_Index'].apply(classify_tier)

print("Index successfully generated!")
print("\nTier Breakdown:")
print(df['Contamination_Tier'].value_counts())

lat_col = [c for c in df.columns if 'lat' in c.lower()][0] if len([c for c in df.columns if 'lat' in c.lower()]) > 0 else 'Latitude'
lon_col = [c for c in df.columns if 'lon' in c.lower() or 'lng' in c.lower()][0] if len([c for c in df.columns if 'lon' in c.lower() or 'lng' in c.lower()]) > 0 else 'Longitude'

print("\nTop 5 Most Contaminated Geographic Zones:")
display(df[[lat_col, lon_col, target_col, 'dist_to_nearest_hotspot_km', 'Composite_Contamination_Index', 'Contamination_Tier']].sort_values(by='Composite_Contamination_Index', ascending=False).head())


In [8]:

df.to_parquet('../data/processed/model_ready_data.parquet', index=False)
print("Saved successfully to data/processed directory!")
